# Trouble-shooting the AI-Synbio LIMS Mirror DB: Data integrity, syncing operations, archiving

Want to be able to 
- Create db from Google Sheets.
- Sync sheets with db
    - schema changes are implemented
    - updated rows are updated, not inserted
    - new rows are inserted
    - deleted rows are marked as deleted
- Sync daemon syncs at defined intervals
- Make copies of the database at defined intervals (archives)
    - Create a copy every (24 hours)
    - Retain only copies at defined intervals and delete all other intermittent copies using archive daemon

## What I have observed to be broken/missing
- At each sync, all rows are re-inserted into the db, even if they haven't changed (and their row hash hasn't changed) resulting in a gargantuan db. I killed the daemon.
- There are problems with file paths to the sync.log which the config defines as /storage/synbio/sync.log. It writes to that location, but for some reason, somewhere else in the code it expects sync.log to be in folder that contains the sync module.
- Starting and stopping the sync daemon doesn't immediately take place, and errors may occur that don't change the global status of _daemon_running.
- I want to have control over the daemon processes and kill them if necessary by hand. There have been times when I was thinking that there might be multiple sync daemon running simultaneously, all writing to the same database, log file - causing problems.
- Automatic copying of the mirror db and cleanup of the archive at the specified intervals is not implemented.
  
## How to fix things - with AI??

- Merge dev into main. Push to GitHub. Pull to laptop.
- Ask copilot:
  - How does the scheduler underlying the sync daemon work? With the current code version, is it theoretically possible to have multiple sync daemons running simultaneously? How can I ensure that this never happens?
  - During sync, all LIMS rows are re-inserted causing the db to balloon. They shouldn't be if the row hash is the same. But they are nonetheless. Why? How can this be prevented?
  - If starting or stopping the sync daemon with start_sync_daemon or stop_sync_daemon() run into an error, the status of the global _daemon_running variable does not necessarily change. How do I ensure the _daemon_running variable represents the actual daemon state?
  - Create an archive daemon (similar to the sync daemon) that makes copies of the database at a specified location and at specified intervals and keeps only those copies specified in the retention policy.




In [ ]:
## Make sure running in aisynbio_env

In [5]:
# Reload magic command to ensure that changes made to my imported modules are being picked up by the notebook continuously

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import sys
import os

# Add project root to path for access to workflows and tasks
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [3]:
import os

# Must add environment's bin to the PATH inside the notebook
env_bin = os.path.join(os.path.dirname(sys.executable), "")
os.environ["PATH"] = env_bin + ":" + os.environ["PATH"]

In [19]:
from aisynbiopipeline.limsapi.query import query_table
import pandas as pd

pd.DataFrame(query_table('Experiments'))

,Database_ID,Name,Type,Protocol,Start_timestamp,Description,Report,Error,deleted,last_synced,row_hash
0,,ALE1b,robotic ALE,ALE1b sample processing,4/4/2025,ALE1b was designed as a proof-of-principle tes...,ALE1b Report,,0,2025-11-14T23:45:28.127262,12b692454937a8256e7ed82c809b3c9f9c64b7db471f9e...
1,,strain_stocks,strain stock testing,NA,5/3/2024,Mock experiment that serves as collection of a...,,,0,2025-11-14T23:45:28.127721,b4500a63abc27a8d2c9d7616a46926f61759e727d4dd51...
2,,TFMN1,robotic mutant competition assay,TFMN1_protocol,10/8/2025,,,incomplete,0,2025-11-14T23:45:28.127755,729503566015225f4430b0f06cf8f4ad4f608b936498bc...
3,,DGOA_EASy,EASy and EASy isolates,EASy_protocol,4/1/2024,Prototrophic selection of strains with dgoA al...,,,0,2025-11-14T23:45:28.127781,9d1b1cb9ef02ce144c7a245c99711b31933c4dcf847041...
4,,ANL_UGA_prot_1,inter lab comparison,,,Comparison of proteomics results from ADP1 sam...,,incomplete,0,2025-11-14T23:45:28.127804,ee6fe35188ca21ab662e40885c83faceb38edf94cd15aa...
5,,ANL_UGA_prot_2,inter lab comparison,,3/26/2025,Repeat experiment of ANL_UGA_prot_1 after stan...,,incomplete,0,2025-11-14T23:45:28.127831,549ebfba9657ff2af1d1e26aa7a411ec51963eb5eac8bf...


In [21]:
from aisynbiopipeline.limsapi.sync import sync_all_sheets

sync_all_sheets()

2025-12-19 16:34:34,691 - lims_sync - INFO - Starting sync operation
2025-12-19 16:34:34,692 - lims_sync - INFO - Connecting to Google Sheets
2025-12-19 16:34:35,645 - lims_sync - INFO - Connecting to database
2025-12-19 16:34:35,817 - lims_sync - INFO - Found 13 worksheets: Experiments, Strains, Conditions, Samples, Measurements, Genes, Measurement_types, DNA_constructs, Primers, dgoA_alleles_new, dgoA_alleles_old, robotic_mt_samples, Strain_stocks_ANL
2025-12-19 16:34:35,818 - lims_sync - INFO - Syncing worksheet: Experiments
2025-12-19 16:34:36,314 - lims_sync - INFO - Retrieved 9 rows from Experiments
2025-12-19 16:34:36,359 - lims_sync - INFO - Inserted 9, updated 0 rows in Experiments
2025-12-19 16:34:41,366 - lims_sync - INFO - Syncing worksheet: Strains
2025-12-19 16:34:41,904 - lims_sync - INFO - Retrieved 1020 rows from Strains
2025-12-19 16:34:41,960 - lims_sync - INFO - Inserted 1020, updated 0 rows in Strains
2025-12-19 16:34:46,985 - lims_sync - INFO - Syncing worksheet: 

{'start_time': '2025-12-19T16:34:34.691809',
 'end_time': '2025-12-19T16:35:49.294381',
 'success': True,
 'tables_synced': 12,
 'total_rows_inserted': 5794,
 'total_rows_updated': 0,
 'total_rows_deleted': 57,
 'errors': ["Error syncing worksheet robotic_mt_samples: Failed to get worksheet data: the header row in the worksheet contains duplicates: ['']To manually set the header row, use the `expected_headers` parameter of `get_all_records()`"]}

In [22]:
from aisynbiopipeline.limsapi.query import query_table
import pandas as pd

pd.DataFrame(query_table('Experiments'))

,Database_ID,Name,Type,Protocol,Start_timestamp,Description,Report,Error,deleted,last_synced,row_hash
0,,ALE1b,robotic ALE,ALE1b sample processing,4/4/2025,ALE1b was designed as a proof-of-principle tes...,ALE1b Report,,0,2025-11-14T23:45:28.127262,12b692454937a8256e7ed82c809b3c9f9c64b7db471f9e...
1,,strain_stocks,strain stock testing,NA,5/3/2024,Mock experiment that serves as collection of a...,,,0,2025-11-14T23:45:28.127721,b4500a63abc27a8d2c9d7616a46926f61759e727d4dd51...
2,,TFMN1,robotic mutant competition assay,TFMN1_protocol,10/8/2025,,,incomplete,0,2025-11-14T23:45:28.127755,729503566015225f4430b0f06cf8f4ad4f608b936498bc...
3,,DGOA_EASy,EASy and EASy isolates,EASy_protocol,4/1/2024,Prototrophic selection of strains with dgoA al...,,,0,2025-11-14T23:45:28.127781,9d1b1cb9ef02ce144c7a245c99711b31933c4dcf847041...
4,,ANL_UGA_prot_1,inter lab comparison,,,Comparison of proteomics results from ADP1 sam...,,incomplete,0,2025-11-14T23:45:28.127804,ee6fe35188ca21ab662e40885c83faceb38edf94cd15aa...
5,,ANL_UGA_prot_2,inter lab comparison,,3/26/2025,Repeat experiment of ANL_UGA_prot_1 after stan...,,incomplete,0,2025-11-14T23:45:28.127831,549ebfba9657ff2af1d1e26aa7a411ec51963eb5eac8bf...
6,,ALE1b,robotic ALE,ALE1b sample processing,4/4/2025,ALE1b was designed as a proof-of-principle tes...,ALE1b Report,,0,2025-12-19T16:34:36.321771,12b692454937a8256e7ed82c809b3c9f9c64b7db471f9e...
7,,strain_stocks,strain stock testing,NA,5/3/2024,Mock experiment that serves as collection of a...,,,0,2025-12-19T16:34:36.327921,b4500a63abc27a8d2c9d7616a46926f61759e727d4dd51...
8,,TFMN1,robotic mutant competition assay,TFMN1_protocol,10/8/2025,,,incomplete,0,2025-12-19T16:34:36.327986,729503566015225f4430b0f06cf8f4ad4f608b936498bc...
9,,DGOA_EASy,EASy and EASy isolates,EASy_protocol,4/1/2024,Prototrophic selection of strains with dgoA al...,,,0,2025-12-19T16:34:36.328025,9d1b1cb9ef02ce144c7a245c99711b31933c4dcf847041...


In [23]:
# Start daemon sync
from aisynbiopipeline.limsapi.sync import start_sync_daemon

start_sync_daemon()

2025-12-19 16:40:03,132 - lims_sync - INFO - Sync daemon started with 10 minute interval
2025-12-19 16:40:03,147 - lims_sync - INFO - Starting sync operation
2025-12-19 16:40:03,148 - lims_sync - INFO - Connecting to Google Sheets
2025-12-19 16:40:03,610 - lims_sync - INFO - Connecting to database
2025-12-19 16:40:03,696 - lims_sync - INFO - Found 13 worksheets: Experiments, Strains, Conditions, Samples, Measurements, Genes, Measurement_types, DNA_constructs, Primers, dgoA_alleles_new, dgoA_alleles_old, robotic_mt_samples, Strain_stocks_ANL
2025-12-19 16:40:03,697 - lims_sync - INFO - Syncing worksheet: Experiments
2025-12-19 16:40:04,231 - lims_sync - INFO - Retrieved 9 rows from Experiments
2025-12-19 16:40:04,252 - lims_sync - INFO - Inserted 9, updated 0 rows in Experiments
2025-12-19 16:40:09,259 - lims_sync - INFO - Syncing worksheet: Strains
2025-12-19 16:40:09,991 - lims_sync - INFO - Retrieved 1020 rows from Strains
2025-12-19 16:40:10,042 - lims_sync - INFO - Inserted 1020, u

In [25]:
from aisynbiopipeline.limsapi.sync import _daemon_running, _daemon_lock_fd, _sync_status

print(_daemon_running)
print(_daemon_lock_fd)
print(_sync_status)

True
57
{'last_sync': '2025-12-19T16:35:49.294381', 'last_success': '2025-12-19T16:35:49.294381', 'last_error': None, 'syncs_completed': 1, 'syncs_failed': 0}


In [26]:
# Stop daemon sync
from aisynbiopipeline.limsapi.sync import stop_sync_daemon

stop_sync_daemon()

In [27]:
from aisynbiopipeline.limsapi.sync import _daemon_running, _daemon_lock_fd, _sync_status

print(_daemon_running)
print(_daemon_lock_fd)
print(_sync_status)

False
None
{'last_sync': '2025-12-19T16:41:17.532795', 'last_success': '2025-12-19T16:41:17.532795', 'last_error': None, 'syncs_completed': 2, 'syncs_failed': 0}


Turned on sync daemon via command line:
(aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ ./lims.sh daemon start

Should not be able to start it from this notebook:

In [28]:
# Start daemon sync
from aisynbiopipeline.limsapi.sync import start_sync_daemon

start_sync_daemon()

2025-12-19 16:49:11,204 - lims_sync - INFO - Sync daemon started with 10 minute interval
2025-12-19 16:49:11,208 - lims_sync - INFO - Starting sync operation
2025-12-19 16:49:11,208 - lims_sync - INFO - Connecting to Google Sheets
2025-12-19 16:49:12,148 - lims_sync - INFO - Connecting to database
2025-12-19 16:49:12,244 - lims_sync - INFO - Found 13 worksheets: Experiments, Strains, Conditions, Samples, Measurements, Genes, Measurement_types, DNA_constructs, Primers, dgoA_alleles_new, dgoA_alleles_old, robotic_mt_samples, Strain_stocks_ANL
2025-12-19 16:49:12,245 - lims_sync - INFO - Syncing worksheet: Experiments
2025-12-19 16:49:12,722 - lims_sync - INFO - Retrieved 9 rows from Experiments
2025-12-19 16:49:12,743 - lims_sync - INFO - Inserted 9, updated 0 rows in Experiments
2025-12-19 16:49:17,751 - lims_sync - INFO - Syncing worksheet: Strains
2025-12-19 16:49:18,289 - lims_sync - INFO - Retrieved 1020 rows from Strains
2025-12-19 16:49:18,352 - lims_sync - INFO - Inserted 1020, u

Apparently, the CLI is not working. 

In [29]:
from aisynbiopipeline.limsapi.sync import _daemon_running, _daemon_lock_fd, _sync_status

print(_daemon_running)
print(_daemon_lock_fd)
print(_sync_status)

True
57
{'last_sync': '2025-12-19T16:51:39.858570', 'last_success': '2025-12-19T16:51:39.858570', 'last_error': None, 'syncs_completed': 4, 'syncs_failed': 0}


In [30]:
# Stop daemon sync
from aisynbiopipeline.limsapi.sync import stop_sync_daemon

stop_sync_daemon()

In [31]:
from aisynbiopipeline.limsapi.sync import _daemon_running, _daemon_lock_fd, _sync_status

print(_daemon_running)
print(_daemon_lock_fd)
print(_sync_status)

False
None
{'last_sync': '2025-12-19T16:51:39.858570', 'last_success': '2025-12-19T16:51:39.858570', 'last_error': None, 'syncs_completed': 4, 'syncs_failed': 0}


In [32]:
# Stop daemon sync
from aisynbiopipeline.limsapi.sync import stop_sync_daemon

stop_sync_daemon()

Exception: Sync daemon is not running

Starting sync daemon in helper notebook. Will check here if I can run a second daemon simultaneously

In [33]:
# Start daemon sync
from aisynbiopipeline.limsapi.sync import start_sync_daemon

start_sync_daemon()

2025-12-19 17:07:10,900 - lims_sync - WARNING - Daemon lock is held by process 4148835


Exception: Sync daemon is already running (locked by another process)

At least this worked. Will check if I can turn it off in here and confirm that it's off in the other notebook

In [34]:
# Stop daemon sync
from aisynbiopipeline.limsapi.sync import stop_sync_daemon

stop_sync_daemon()

Exception: Sync daemon is not running

In [35]:
from aisynbiopipeline.limsapi.sync import _daemon_running, _daemon_lock_fd, _sync_status

print(_daemon_running)
print(_daemon_lock_fd)
print(_sync_status)

False
None
{'last_sync': '2025-12-19T16:51:39.858570', 'last_success': '2025-12-19T16:51:39.858570', 'last_error': None, 'syncs_completed': 4, 'syncs_failed': 0}


Cannot stop it from this notebook, even though it is running. Will try to restart:

In [36]:
# Start daemon sync
from aisynbiopipeline.limsapi.sync import start_sync_daemon

start_sync_daemon()

2025-12-19 17:11:13,705 - lims_sync - WARNING - Daemon lock is held by process 4148835


Exception: Sync daemon is already running (locked by another process)

In [38]:
from aisynbiopipeline.limsapi.sync import _daemon_running, _daemon_lock_fd, _sync_status

print(_daemon_running)
print(_daemon_lock_fd)
print(_sync_status)

False
None
{'last_sync': '2025-12-19T16:51:39.858570', 'last_success': '2025-12-19T16:51:39.858570', 'last_error': None, 'syncs_completed': 4, 'syncs_failed': 0}


In [39]:
# Start daemon sync
from aisynbiopipeline.limsapi.sync import start_sync_daemon

start_sync_daemon()

2025-12-19 17:24:19,661 - lims_sync - WARNING - Daemon lock is held by process 4148835


Exception: Sync daemon is already running (locked by another process)

In [40]:
from aisynbiopipeline.limsapi.query import query_table
import pandas as pd

pd.DataFrame(query_table('Experiments'))

,Database_ID,Name,Type,Protocol,Start_timestamp,Description,Report,Error,deleted,last_synced,row_hash
0,,ALE1b,robotic ALE,ALE1b sample processing,4/4/2025,ALE1b was designed as a proof-of-principle tes...,ALE1b Report,,0,2025-11-14T23:45:28.127262,12b692454937a8256e7ed82c809b3c9f9c64b7db471f9e...
1,,strain_stocks,strain stock testing,NA,5/3/2024,Mock experiment that serves as collection of a...,,,0,2025-11-14T23:45:28.127721,b4500a63abc27a8d2c9d7616a46926f61759e727d4dd51...
2,,TFMN1,robotic mutant competition assay,TFMN1_protocol,10/8/2025,,,incomplete,0,2025-11-14T23:45:28.127755,729503566015225f4430b0f06cf8f4ad4f608b936498bc...
3,,DGOA_EASy,EASy and EASy isolates,EASy_protocol,4/1/2024,Prototrophic selection of strains with dgoA al...,,,0,2025-11-14T23:45:28.127781,9d1b1cb9ef02ce144c7a245c99711b31933c4dcf847041...
4,,ANL_UGA_prot_1,inter lab comparison,,,Comparison of proteomics results from ADP1 sam...,,incomplete,0,2025-11-14T23:45:28.127804,ee6fe35188ca21ab662e40885c83faceb38edf94cd15aa...
5,,ANL_UGA_prot_2,inter lab comparison,,3/26/2025,Repeat experiment of ANL_UGA_prot_1 after stan...,,incomplete,0,2025-11-14T23:45:28.127831,549ebfba9657ff2af1d1e26aa7a411ec51963eb5eac8bf...


In [41]:
from aisynbiopipeline.limsapi.sync import sync_all_sheets

sync_all_sheets()

2025-12-19 21:50:23,980 - lims_sync - INFO - Starting sync operation
2025-12-19 21:50:23,981 - lims_sync - INFO - Connecting to Google Sheets
2025-12-19 21:50:24,981 - lims_sync - INFO - Connecting to database
2025-12-19 21:50:25,090 - lims_sync - INFO - Found 13 worksheets: Experiments, Strains, Conditions, Samples, Measurements, Genes, Measurement_types, DNA_constructs, Primers, dgoA_alleles_new, dgoA_alleles_old, robotic_mt_samples, Strain_stocks_ANL
2025-12-19 21:50:25,091 - lims_sync - INFO - Syncing worksheet: Experiments
2025-12-19 21:50:25,627 - lims_sync - INFO - Retrieved 9 rows from Experiments
2025-12-19 21:50:25,701 - lims_sync - INFO - Inserted 3, updated 0 rows in Experiments
2025-12-19 21:50:30,707 - lims_sync - INFO - Syncing worksheet: Strains
2025-12-19 21:50:31,321 - lims_sync - INFO - Retrieved 1020 rows from Strains
2025-12-19 21:50:31,415 - lims_sync - INFO - Inserted 28, updated 0 rows in Strains
2025-12-19 21:50:31,437 - lims_sync - INFO - Marked 1020 rows as d

{'start_time': '2025-12-19T21:50:23.980415',
 'end_time': '2025-12-19T21:53:24.390010',
 'success': True,
 'tables_synced': 12,
 'total_rows_inserted': 410,
 'total_rows_updated': 0,
 'total_rows_deleted': 1077,
 'errors': ["Error syncing worksheet robotic_mt_samples: Failed to get worksheet data: the header row in the worksheet contains duplicates: ['']To manually set the header row, use the `expected_headers` parameter of `get_all_records()`"]}

In [47]:
from aisynbiopipeline.limsapi.query import query_table
import pandas as pd

experiments = pd.DataFrame(query_table('Experiments'))
experiments

,Database_ID,Name,Type,Protocol,Start_timestamp,Description,Report,Error,deleted,last_synced,row_hash
0,,ALE1b,robotic ALE,ALE1b sample processing,4/4/2025,ALE1b was designed as a proof-of-principle tes...,ALE1b Report,,0,2025-11-14T23:45:28.127262,12b692454937a8256e7ed82c809b3c9f9c64b7db471f9e...
1,,strain_stocks,strain stock testing,NA,5/3/2024,Mock experiment that serves as collection of a...,,,0,2025-11-14T23:45:28.127721,b4500a63abc27a8d2c9d7616a46926f61759e727d4dd51...
2,,TFMN1,robotic mutant competition assay,TFMN1_protocol,10/8/2025,,,incomplete,0,2025-11-14T23:45:28.127755,729503566015225f4430b0f06cf8f4ad4f608b936498bc...
3,,DGOA_EASy,EASy and EASy isolates,EASy_protocol,4/1/2024,Prototrophic selection of strains with dgoA al...,,,0,2025-11-14T23:45:28.127781,9d1b1cb9ef02ce144c7a245c99711b31933c4dcf847041...
4,,ANL_UGA_prot_1,inter lab comparison,,,Comparison of proteomics results from ADP1 sam...,,incomplete,0,2025-11-14T23:45:28.127804,ee6fe35188ca21ab662e40885c83faceb38edf94cd15aa...
5,,ANL_UGA_prot_2,inter lab comparison,,3/26/2025,Repeat experiment of ANL_UGA_prot_1 after stan...,,incomplete,0,2025-11-14T23:45:28.127831,549ebfba9657ff2af1d1e26aa7a411ec51963eb5eac8bf...
6,,ANL_prot_3,proteomics standardization,,12/9/2025,Comparison of proteomics results for multiple ...,,incomplete,0,2025-12-19T21:50:25.646543,20e663b9f747d30e643d4ada5d6b463aba614eb80adb33...
7,,ANL_prot_4,proteome characterization,,12/16/2025,Characterizing the proteome of ACN3560 (colony...,,incomplete,0,2025-12-19T21:50:25.694937,730c7cc8665b443dcc8e4bcc38889fdc63170b31e43d74...
8,,TFMN2,robotic mutant competition assay,TFMN2_protocol,,,,incomplete,0,2025-12-19T21:50:25.695025,29ed7d1a19820528fd34d5ac9aeb8e535e87acea21f2fc...


In [48]:
len(experiments)

9

In [44]:
from aisynbiopipeline.limsapi.query import query_table
import pandas as pd

samples = pd.DataFrame(query_table('Samples'))
samples.loc[samples['Name']== '']

,Database_ID,Name,Experiment,Type,Condition,Strain_name,Transforming_DNA,Protocol,Parent_sample,Replicate_samples,Innoculation_timestamp,Measurements,Notes,Error,deleted,last_synced,row_hash
125,,,,,,,,,,,,,,incomplete,0,2025-12-19T21:50:42.573175,f65ec444a20710d6b23d7640e50fcd4e02f2f0c186ae90...


In [45]:
len(samples)

126

In [46]:
samples.Name.nunique()

126

In [49]:
from aisynbiopipeline.limsapi.sync import sync_all_sheets

sync_all_sheets()

2025-12-19 21:58:31,677 - lims_sync - INFO - Starting sync operation
2025-12-19 21:58:31,678 - lims_sync - INFO - Connecting to Google Sheets
2025-12-19 21:58:32,197 - lims_sync - INFO - Connecting to database
2025-12-19 21:58:32,436 - lims_sync - INFO - Found 13 worksheets: Experiments, Strains, Conditions, Samples, Measurements, Genes, Measurement_types, DNA_constructs, Primers, dgoA_alleles_new, dgoA_alleles_old, robotic_mt_samples, Strain_stocks_ANL
2025-12-19 21:58:32,437 - lims_sync - INFO - Syncing worksheet: Experiments
2025-12-19 21:58:33,171 - lims_sync - INFO - Retrieved 9 rows from Experiments
2025-12-19 21:58:33,199 - lims_sync - INFO - Inserted 0, updated 0 rows in Experiments
2025-12-19 21:58:38,204 - lims_sync - INFO - Syncing worksheet: Strains
2025-12-19 21:58:38,956 - lims_sync - INFO - Retrieved 1020 rows from Strains
2025-12-19 21:58:42,051 - lims_sync - INFO - Inserted 0, updated 0 rows in Strains
2025-12-19 21:58:47,080 - lims_sync - INFO - Syncing worksheet: Con

{'start_time': '2025-12-19T21:58:31.677632',
 'end_time': '2025-12-19T22:01:38.067875',
 'success': True,
 'tables_synced': 12,
 'total_rows_inserted': 0,
 'total_rows_updated': 0,
 'total_rows_deleted': 0,
 'errors': ["Error syncing worksheet robotic_mt_samples: Failed to get worksheet data: the header row in the worksheet contains duplicates: ['']To manually set the header row, use the `expected_headers` parameter of `get_all_records()`"]}